# DreamVision 2.0: HiPerGator Background GAN (`ImageFolder`)

This notebook trains the background GAN on your curated HiPerGator subset using `torchvision.datasets.ImageFolder`.

Expected uploaded dataset:
```text
data_256/
  bedroom/
  forest_path/
  kitchen/
  living_room/
  park/
  street/
```


## Plan

Start with `forest_path` only so we can learn one coherent world cleanly at `256x256`. Once the outputs look good, expand the selected class list to include `park`, `street`, or the full uploaded subset.


In [ ]:
from __future__ import annotations

import random
import time
from dataclasses import dataclass, asdict
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import ImageFile
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, utils
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
print(torch.__version__)


In [ ]:
@dataclass
class HPGConfig:
    data_root: Path = Path("/blue/<group>/<username>/data_256")
    output_root: Path = Path("/blue/<group>/<username>/dreamvision_outputs/backgrounds_256")
    checkpoint_dir: Path = output_root / "checkpoints"
    sample_dir: Path = output_root / "samples"
    image_size: int = 256
    batch_size: int = 16
    num_workers: int = 8
    latent_dim: int = 256
    ngf: int = 64
    ndf: int = 64
    num_channels: int = 3
    num_epochs: int = 30
    learning_rate_g: float = 1e-4
    learning_rate_d: float = 2e-4
    beta1: float = 0.5
    sample_interval: int = 1
    checkpoint_interval: int = 5
    selected_classes: list[str] | None = None
    max_images_per_class: int | None = None
    max_steps_per_epoch: int | None = None
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


config = HPGConfig()
config.selected_classes = ["forest_path"]
config.checkpoint_dir.mkdir(parents=True, exist_ok=True)
config.sample_dir.mkdir(parents=True, exist_ok=True)
print(asdict(config))


## First Edit

Before running, update `config.data_root` and `config.output_root` to your real HiPerGator paths.


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


set_seed(config.seed)
device = torch.device(config.device)
device


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize(int(config.image_size * 1.1)),
    transforms.CenterCrop(config.image_size),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

base_dataset = datasets.ImageFolder(root=str(config.data_root), transform=train_transform)
print("Discovered classes:", base_dataset.classes)

selected_class_ids = [base_dataset.class_to_idx[name] for name in config.selected_classes]
selected_class_set = set(selected_class_ids)

class_buckets = {class_id: [] for class_id in selected_class_ids}
for dataset_index, (_, target) in enumerate(base_dataset.samples):
    if target in selected_class_set:
        if config.max_images_per_class is None or len(class_buckets[target]) < config.max_images_per_class:
            class_buckets[target].append(dataset_index)

subset_indices = []
for class_id in selected_class_ids:
    subset_indices.extend(class_buckets[class_id])

random.Random(config.seed).shuffle(subset_indices)
train_dataset = Subset(base_dataset, subset_indices)
train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=(device.type == "cuda"),
    drop_last=True,
    persistent_workers=config.num_workers > 0,
)

print("Selected classes:")
for class_name in config.selected_classes:
    class_id = base_dataset.class_to_idx[class_name]
    print(f"- {class_name}: {len(class_buckets[class_id])} images")

print(f"Total subset size: {len(train_dataset):,}")
print(f"Batches per epoch: {len(train_loader):,}")


In [ ]:
def denormalize(images: torch.Tensor) -> torch.Tensor:
    return (images * 0.5 + 0.5).clamp(0, 1)

batch_images, _ = next(iter(train_loader))
grid = utils.make_grid(denormalize(batch_images[:4]), nrow=2)
plt.figure(figsize=(8, 8))
plt.axis("off")
plt.imshow(grid.permute(1, 2, 0))
plt.show()


In [ ]:
class UpBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="nearest"),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(True),
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)

class Generator(nn.Module):
    def __init__(self, latent_dim: int, ngf: int, num_channels: int) -> None:
        super().__init__()
        self.project = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, ngf * 16, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 16),
            nn.ReLU(True),
        )
        self.upsampler = nn.Sequential(
            UpBlock(ngf * 16, ngf * 8),
            UpBlock(ngf * 8, ngf * 4),
            UpBlock(ngf * 4, ngf * 2),
            UpBlock(ngf * 2, ngf),
            UpBlock(ngf, ngf // 2),
            UpBlock(ngf // 2, ngf // 4),
        )
        self.to_rgb = nn.Sequential(
            nn.Conv2d(ngf // 4, num_channels, kernel_size=3, stride=1, padding=1),
            nn.Tanh(),
        )
    def forward(self, noise: torch.Tensor) -> torch.Tensor:
        x = self.project(noise)
        x = self.upsampler(x)
        return self.to_rgb(x)

class Discriminator(nn.Module):
    def __init__(self, ndf: int, num_channels: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(num_channels, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 8, ndf * 16, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 16),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 16, ndf * 32, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 32),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 32, 1, 4, 1, 0, bias=False),
        )
    def forward(self, images: torch.Tensor) -> torch.Tensor:
        return self.net(images).view(-1)

def weights_init(module: nn.Module) -> None:
    classname = module.__class__.__name__
    if "Conv" in classname:
        nn.init.normal_(module.weight.data, 0.0, 0.02)
    elif "BatchNorm" in classname:
        nn.init.normal_(module.weight.data, 1.0, 0.02)
        nn.init.constant_(module.bias.data, 0)


In [ ]:
generator = Generator(config.latent_dim, config.ngf, config.num_channels).to(device)
discriminator = Discriminator(config.ndf, config.num_channels).to(device)
generator.apply(weights_init)
discriminator.apply(weights_init)

criterion = nn.BCEWithLogitsLoss()
optimizer_g = optim.Adam(generator.parameters(), lr=config.learning_rate_g, betas=(config.beta1, 0.999))
optimizer_d = optim.Adam(discriminator.parameters(), lr=config.learning_rate_d, betas=(config.beta1, 0.999))

fixed_noise = torch.randn(4, config.latent_dim, 1, 1, device=device)
print(sum(p.numel() for p in generator.parameters()))
print(sum(p.numel() for p in discriminator.parameters()))


In [ ]:
def save_checkpoint(epoch: int) -> None:
    torch.save({
        "epoch": epoch,
        "config": asdict(config),
        "generator_state_dict": generator.state_dict(),
        "discriminator_state_dict": discriminator.state_dict(),
        "optimizer_g_state_dict": optimizer_g.state_dict(),
        "optimizer_d_state_dict": optimizer_d.state_dict(),
    }, config.checkpoint_dir / f"hpg_background_epoch_{epoch:03d}.pt")

def save_sample_grid(epoch: int) -> Path:
    generator.eval()
    with torch.no_grad():
        samples = generator(fixed_noise).cpu()
    generator.train()
    sample_path = config.sample_dir / f"hpg_background_epoch_{epoch:03d}.png"
    utils.save_image(denormalize(samples), sample_path, nrow=2)
    return sample_path

def show_sample_grid(sample_path: Path, title: str | None = None) -> None:
    image = plt.imread(sample_path)
    plt.figure(figsize=(8, 8))
    if title is not None:
        plt.title(title)
    plt.axis("off")
    plt.imshow(image)
    plt.show()


In [ ]:
history = []

for epoch in range(1, config.num_epochs + 1):
    start_time = time.time()
    g_running = 0.0
    d_running = 0.0
    steps_completed = 0
    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{config.num_epochs}")
    for step_idx, (real_images, _) in enumerate(progress, start=1):
        if config.max_steps_per_epoch is not None and step_idx > config.max_steps_per_epoch:
            break
        real_images = real_images.to(device, non_blocking=True)
        batch_size = real_images.size(0)
        real_targets = torch.ones(batch_size, device=device)
        fake_targets = torch.zeros(batch_size, device=device)
        discriminator.zero_grad(set_to_none=True)
        real_output = discriminator(real_images)
        d_loss_real = criterion(real_output, real_targets)
        noise = torch.randn(batch_size, config.latent_dim, 1, 1, device=device)
        fake_images = generator(noise)
        fake_output = discriminator(fake_images.detach())
        d_loss_fake = criterion(fake_output, fake_targets)
        d_loss = d_loss_real + d_loss_fake
        d_loss.backward()
        optimizer_d.step()
        generator.zero_grad(set_to_none=True)
        gen_output = discriminator(fake_images)
        g_loss = criterion(gen_output, real_targets)
        g_loss.backward()
        optimizer_g.step()
        g_running += g_loss.item()
        d_running += d_loss.item()
        steps_completed += 1
        progress.set_postfix(d_loss=f"{d_loss.item():.4f}", g_loss=f"{g_loss.item():.4f}")
    avg_g = g_running / max(steps_completed, 1)
    avg_d = d_running / max(steps_completed, 1)
    elapsed = time.time() - start_time
    history.append({"epoch": epoch, "g_loss": avg_g, "d_loss": avg_d, "epoch_time_sec": elapsed, "steps_completed": steps_completed})
    print(f"Epoch {epoch:03d} | steps: {steps_completed} | D: {avg_d:.4f} | G: {avg_g:.4f} | time: {elapsed:.1f}s")
    if epoch % config.sample_interval == 0:
        sample_path = save_sample_grid(epoch)
        show_sample_grid(sample_path, title=f"Generated backgrounds after epoch {epoch}")
    if epoch % config.checkpoint_interval == 0:
        save_checkpoint(epoch)


In [ ]:
epochs = [row["epoch"] for row in history]
g_losses = [row["g_loss"] for row in history]
d_losses = [row["d_loss"] for row in history]
plt.figure(figsize=(10, 5))
plt.plot(epochs, g_losses, label="Generator")
plt.plot(epochs, d_losses, label="Discriminator")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("HiPerGator Background GAN Training Loss")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()
